# Lab 11: Movie Recommendations from Similar Users and Similar Movies

**Task:** recommend movies a user hasn't seen yet, based on the ratings of users with similar taste, or on movies similar to the ones they liked.

**Dataset:** MovieLens "latest small" (https://files.grouplens.org/datasets/movielens/ml-latest-small.zip), included as `data/ml-latest-small.zip` as downloaded: 100,836 ratings (0.5 to 5 stars) by 610 users of 9,742 movies.

The idea is the nearest-neighbour idea from k-NN, applied in a new way:
- **user-user**: describe each user by their ratings of all movies; find the users most similar to you (cosine similarity); recommend what they liked that you haven't seen;
- **item-item**: describe each movie by the ratings it got from all users; movies rated similarly by the same people are "similar"; recommend movies similar to the ones you liked.

**What we will do:**
1. Build the user–movie rating matrix.
2. Compute user-user and item-item cosine similarities.
3. Recommend the top-N unseen movies for a chosen user, with explanations ("because you liked ...").
4. Evaluate on held-out ratings: RMSE of predicted ratings, and hit-rate@N of the recommendations.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
z = zipfile.ZipFile("data/ml-latest-small.zip")
print(z.namelist())
ratings = pd.read_csv(z.open("ml-latest-small/ratings.csv"))
movies = pd.read_csv(z.open("ml-latest-small/movies.csv"))
print(ratings.shape, movies.shape)
ratings.head()

In [ ]:
print("users:", ratings["userId"].nunique(), "| movies rated:", ratings["movieId"].nunique())
per_user = ratings.groupby("userId").size()
per_movie = ratings.groupby("movieId").size()
print("ratings per user: min", per_user.min(), "median", int(per_user.median()), "max", per_user.max())
print("ratings per movie: median", int(per_movie.median()), "| movies with a single rating:", (per_movie == 1).sum())
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
ratings["rating"].value_counts().sort_index().plot.bar(ax=axes[0], title="rating values")
per_movie.clip(upper=100).hist(bins=50, ax=axes[1]); axes[1].set_title("ratings per movie (clipped at 100)")
plt.show()

Typical of rating data: most users rate few movies, most movies have few ratings, and a few popular movies have hundreds. The rating matrix (610 users × 9,742 movies) is **98% empty**.

## Holding out ratings for evaluation

To test recommendations honestly we hide 20% of each user's ratings, build everything from the remaining 80%, and check whether the hidden movies are recommended (and how well their ratings are predicted).

In [ ]:
test = ratings.groupby("userId", group_keys=False).sample(frac=0.2, random_state=0)
train = ratings.drop(test.index)
print(len(train), "training ratings,", len(test), "held-out ratings")

## 1. The rating matrix

Rows are users, columns are movies, entries are ratings; most are missing. We store it as a dense table here (610 × 9,000 is small); large systems use sparse matrices.

**Mean-centring.** Some people rate everything 4–5, others are harsh critics. A 4 from a harsh critic means more than a 4 from someone who gives everything a 4. Subtracting each user's average rating turns ratings into "liked more / less than usual", which makes users comparable. Missing ratings become 0, meaning "no information".

In [ ]:
R = train.pivot(index="userId", columns="movieId", values="rating")
user_mean = R.mean(axis=1)
# centred ratings, 0 = not rated. (Done with NumPy: with some combinations of pandas and an old numexpr
# package, R.sub(user_mean, axis=0) on a table this large silently gives wrong results.)
C = pd.DataFrame(R.values - user_mean.values[:, None], index=R.index, columns=R.columns).fillna(0)
print(R.shape, f"filled: {R.notna().mean().mean():.2%}")

## 2. Cosine similarity

The cosine similarity of two vectors is the cosine of the angle between them: $\cos(\mathbf{a}, \mathbf{b}) = \dfrac{\mathbf{a}^T\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$, between −1 and 1. On centred ratings it is close to 1 when two users liked and disliked the same movies, near 0 when they have nothing in common, negative when their tastes are opposite. It ignores the length of the vectors, so a user who rated 500 movies can still be similar to one who rated 20.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_sim = pd.DataFrame(cosine_similarity(C.values), index=C.index, columns=C.index)
item_sim = pd.DataFrame(cosine_similarity(C.values.T), index=C.columns, columns=C.columns)
np.fill_diagonal(user_sim.values, 0)
np.fill_diagonal(item_sim.values, 0)
print("user-user:", user_sim.shape, "| item-item:", item_sim.shape)

title = movies.set_index("movieId")["title"]
toy_story = 1
print("Movies most similar to", title[toy_story])
print(item_sim[toy_story].nlargest(8).rename(title).round(3))

Toy Story's closest neighbours are its two sequels, followed by family classics (Back to the Future, Mary Poppins, It's a Wonderful Life): without any information about genres, the ratings alone revealed that the same people like these movies. A few unexpected titles (True Grit, Goodfellas) show that similarities estimated from a few hundred users are noisy.

A caution about similarities computed from few ratings: two movies each rated by the same two users can come out "perfectly similar" by chance. We will only recommend movies with enough ratings.

## 3. Predicting ratings and recommending

**User-user prediction** of user $u$'s rating of movie $m$: take the $k$ most similar users who rated $m$, and average their centred ratings of $m$, weighted by similarity; add back $u$'s average:
$$\hat{r}_{um} = \bar{r}_u + \frac{\sum_{v} \text{sim}(u, v)\,(r_{vm} - \bar{r}_v)}{\sum_v |\text{sim}(u, v)|}$$
**Item-item prediction**: the same with the roles swapped: average $u$'s own centred ratings of the $k$ movies most similar to $m$.

In [ ]:
def predict_user_based(u, m, k=30):
    if m not in R.columns:
        return user_mean[u]
    raters = R[m].dropna().index.drop(u, errors="ignore")
    sims = user_sim.loc[u, raters]
    top = sims[sims > 0].nlargest(k)
    if top.empty:
        return user_mean[u]
    return user_mean[u] + (top * C.loc[top.index, m]).sum() / top.abs().sum()

def predict_item_based(u, m, k=30):
    if m not in R.columns:
        return user_mean[u]
    rated = R.loc[u].dropna().index
    sims = item_sim.loc[m, rated]
    top = sims[sims > 0].nlargest(k)
    if top.empty:
        return user_mean[u]
    return user_mean[u] + (top * C.loc[u, top.index]).sum() / top.abs().sum()

### Top-N recommendations for one user, with explanations

In [ ]:
popular_enough = per_movie[per_movie >= 20].index.intersection(R.columns)    # only recommend movies with at least 20 ratings

def recommend(u, n=10, kind="item"):
    seen = R.loc[u].dropna().index
    candidates = popular_enough.difference(seen)
    predict = predict_item_based if kind == "item" else predict_user_based
    scores = pd.Series({m: predict(u, m) for m in candidates})
    return scores.nlargest(n)

user = 42
liked = R.loc[user].dropna().sort_values(ascending=False)
print(f"User {user} rated {len(liked)} movies; favourites:")
print(liked.head(6).rename(title).to_string())
recs = recommend(user, kind="item")
print("\nItem-item recommendations:")
for m, score in recs.items():
    because = item_sim.loc[m, liked[liked >= 4].index].nlargest(2)
    print(f"  {title[m]:50s} predicted {score:.1f}  | because you liked: {', '.join(title[b] for b in because.index)}")

In [ ]:
recs_u = recommend(user, kind="user")
neighbours = user_sim.loc[user].nlargest(5).index
print("User-user recommendations (from users", list(neighbours), "and others):")
for m, score in recs_u.items():
    fans = [v for v in neighbours if R.loc[v, m] >= 4] if m in R.columns else []
    print(f"  {title[m]:50s} predicted {score:.1f}  | rated 4+ by similar users: {fans[:3]}")

## 4. Evaluation

### RMSE of predicted ratings
For a sample of held-out ratings, compare predictions with the real ratings. Baselines: the user's own average rating, and the movie's average rating.

In [ ]:
sample = test.sample(4000, random_state=1)
movie_mean = train.groupby("movieId")["rating"].mean()
preds = {
    "user's average": [user_mean[u] for u in sample["userId"]],
    "movie's average": [movie_mean.get(m, user_mean[u]) for u, m in zip(sample["userId"], sample["movieId"])],
    "user-user (k=30)": [predict_user_based(u, m) for u, m in zip(sample["userId"], sample["movieId"])],
    "item-item (k=30)": [predict_item_based(u, m) for u, m in zip(sample["userId"], sample["movieId"])],
}
rmse = {name: np.sqrt(np.mean((np.clip(p, 0.5, 5) - sample["rating"].values) ** 2)) for name, p in preds.items()}
pd.Series(rmse).round(4).rename("RMSE on held-out ratings")

### Hit-rate@10
RMSE measures rating accuracy, but users see a *list*. **Hit-rate@N**: for each user, recommend N movies they haven't rated (in the training data); it's a hit if at least one of them is among the movies they actually rated highly (4 or more) in the held-out data. A popularity baseline, recommending the most-rated movies the user hasn't seen, is the bar to beat.

In [ ]:
liked_test = test[test["rating"] >= 4].groupby("userId")["movieId"].apply(set)
eval_users = liked_test.index[:150]
most_popular = per_movie.loc[popular_enough].sort_values(ascending=False).index

def hit_rate(recommender, n=10):
    hits = 0
    for u in eval_users:
        hits += len(set(recommender(u, n)) & liked_test[u]) > 0
    return hits / len(eval_users)

def popularity(u, n):
    seen = set(R.loc[u].dropna().index)
    return [m for m in most_popular if m not in seen][:n]

results = {"most popular unseen": hit_rate(popularity),
           "user-user": hit_rate(lambda u, n: recommend(u, n, "user").index),
           "item-item": hit_rate(lambda u, n: recommend(u, n, "item").index)}
pd.Series(results).round(3).rename("hit-rate@10")

### What the evaluation shows

- **On rating accuracy (RMSE)**, both neighbour methods beat the averages: knowing how similar users rated a movie, or how you rated similar movies, says more than averages do.
- **On hit-rate**, the simple popularity list is a very strong baseline, often stronger than the neighbour methods. Popular movies are popular because many people watch and like them, so the held-out "liked" movies are disproportionately popular ones. The neighbour methods score obscure movies with few ratings highly when one or two similar users loved them; that is good for discovery, but it often misses.
- **The two metrics disagree**, which is common: predicting ratings well is not the same as ranking the right movies at the top. A recommender should be judged on the metric that matches how it's used (a top-10 list on a home page), and against a popularity baseline.

**Things to try:** change $k$; change the minimum number of ratings for recommended movies (20 here); blend the neighbour score with popularity.

**Where to go from here.** What we built is known as **collaborative filtering** (recommendations from the behaviour of many users, without knowing anything about the movies themselves). Production recommenders usually go further with **matrix factorisation** methods, which learn a small number of hidden "taste" dimensions for every user and movie from the rating matrix, and generally predict better and scale to millions of users. They are beyond this course; the neighbour methods here are the intuition behind them.